# Notebook 01 — Exploratory Data Analysis (EDA)
## Automated Plant Disease Detection from Leaf Images

This notebook explores the **PlantVillage** dataset:
- Class distribution
- Sample grid visualisation (healthy vs diseased)
- Pixel intensity statistics
- Image resolution analysis
- SimCLR augmentation preview

In [ ]:
import os, sys, random
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from collections import Counter
from torchvision import datasets
from src.dataset import SimCLRAugmentation, get_val_transforms, IMG_SIZE
from src.utils import set_seed, CLASS_NAMES

set_seed(42)
sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.dpi'] = 120

# ─────────────────────────────────────────────────────────
DATA_ROOT = '/Users/aman/Documents/PlantVillage'   # <- change to your path
# ─────────────────────────────────────────────────────────

print('Loading dataset...')
ds = datasets.ImageFolder(DATA_ROOT)
print(f'Total images : {len(ds)}')
print(f'Total classes: {len(ds.classes)}')

## 1. Class Distribution

In [ ]:
label_counts = Counter(label for _, label in ds.samples)
sorted_labels = sorted(label_counts, key=label_counts.get, reverse=True)
sorted_names  = [ds.classes[i].replace('___', ' | ').replace('_', ' ') for i in sorted_labels]
sorted_counts = [label_counts[i] for i in sorted_labels]

fig, ax = plt.subplots(figsize=(14, 10))
colors = ['#2ecc71' if 'healthy' in n else '#e74c3c' for n in sorted_names]
bars = ax.barh(sorted_names, sorted_counts, color=colors, height=0.7)
ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title('PlantVillage Class Distribution\n(Green=Healthy, Red=Diseased)',
             fontsize=14, fontweight='bold')

# Add count labels
for bar, cnt in zip(bars, sorted_counts):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            str(cnt), va='center', ha='left', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total healthy images : {sum(c for n,c in zip(sorted_names,sorted_counts) if "healthy" in n)}')
print(f'Total diseased images: {sum(c for n,c in zip(sorted_names,sorted_counts) if "healthy" not in n)}')

## 2. Sample Grid — Healthy vs Diseased

In [ ]:
def show_sample_grid(dataset, n_cols=6, n_rows=5, figsize=(18, 15)):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    fig.patch.set_facecolor('#1a1a2e')
    sampled_indices = random.sample(range(len(dataset)), n_rows * n_cols)
    for ax, idx in zip(axes.flat, sampled_indices):
        img, label = dataset[idx]
        ax.imshow(img)
        cls_name = dataset.classes[label].replace('___', '\n').replace('_', ' ')
        color = '#2ecc71' if 'healthy' in cls_name else '#e74c3c'
        ax.set_title(cls_name, fontsize=6, color=color, fontweight='bold')
        ax.axis('off')
    plt.suptitle('PlantVillage Sample Images', color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/sample_grid.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

show_sample_grid(ds)

## 3. Pixel Intensity Distribution

In [ ]:
n_sample = 500
sample_indices = random.sample(range(len(ds)), n_sample)
channel_means = {'R': [], 'G': [], 'B': []}

for idx in sample_indices:
    img, _ = ds[idx]
    arr = np.array(img.resize((64, 64)), dtype=np.float32) / 255.0
    channel_means['R'].append(arr[:, :, 0].mean())
    channel_means['G'].append(arr[:, :, 1].mean())
    channel_means['B'].append(arr[:, :, 2].mean())

fig, ax = plt.subplots(figsize=(10, 5))
for ch, color, label in zip(['R', 'G', 'B'], ['#e74c3c', '#2ecc71', '#3498db'], ['Red', 'Green', 'Blue']):
    sns.kdeplot(channel_means[ch], ax=ax, color=color, fill=True, alpha=0.3, label=label)
ax.set_xlabel('Mean Pixel Intensity (normalised)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'RGB Channel Intensity Distribution (n={n_sample} samples)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/pixel_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Image Resolution Statistics

In [ ]:
widths, heights = [], []
for idx in sample_indices:
    img, _ = ds[idx]
    widths.append(img.size[0])
    heights.append(img.size[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].set_title('Image Width Distribution', fontweight='bold')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=30, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1].set_title('Image Height Distribution', fontweight='bold')
axes[1].set_xlabel('Height (px)')

print(f'Width  — mean: {np.mean(widths):.0f}px  std: {np.std(widths):.0f}px')
print(f'Height — mean: {np.mean(heights):.0f}px  std: {np.std(heights):.0f}px')
plt.tight_layout()
plt.savefig('../outputs/resolution_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SimCLR Augmentation Preview
Shows how the same image produces two very different 'views' used for contrastive learning.

In [ ]:
aug = SimCLRAugmentation(size=224)

# Pick 4 random images
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.patch.set_facecolor('#1a1a2e')

for row in range(4):
    idx = random.randint(0, len(ds) - 1)
    img, label = ds[idx]
    v1, v2 = aug(img)

    # denormalise for display
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    def denorm(t):
        arr = t.permute(1, 2, 0).numpy() * std + mean
        return np.clip(arr, 0, 1)

    name = ds.classes[label].replace('___', ' | ').replace('_', ' ')
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'Original\n{name}', color='white', fontsize=8)
    axes[row, 1].imshow(denorm(v1))
    axes[row, 1].set_title('SimCLR View 1', color='#00d2ff', fontsize=8)
    axes[row, 2].imshow(denorm(v2))
    axes[row, 2].set_title('SimCLR View 2', color='#00d2ff', fontsize=8)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('SimCLR Augmentation — Two Views per Image', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/simclr_augmentations.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()